In [ ]:
# ============================================================
# ANNUAL RECHARGE AND DROUGHT-INDEX COMPARISON
#
# Weekly SPI/SPEI data are summarized into yearly means.
#
# Outputs for each discharge station/catchment:
#
# Plot 1:
#   Left y-axis  = Annual WTF recharge (mm)
#   Right y-axis = Yearly mean SPI and SPEI
#
# Plot 2:
#   Left y-axis  = Annual streamflow-based recharge (mm)
#   Right y-axis = Yearly mean SPI and SPEI
#
# Only one final ZIP file is retained.
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import re
import glob
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import FileLink, display


# ============================================================
# 2. CLEAN OLD OUTPUTS
# ============================================================

old_output_folders = [
    "/kaggle/working/annual_recharge_drought_comparison",
    "/kaggle/working/annual_recharge_drought_comparison_corrected",
    "/kaggle/working/annual_recharge_drought_comparison_attractive",
    "/kaggle/working/recharge_comparison",
    "/kaggle/working/recharge_weekly_drought_comparison",
    "/kaggle/working/recharge_drought_comparison",
    "/kaggle/working/single_recharge_drought_plots",
    "/kaggle/working/recharge_correlation_matrices"
]

for folder in old_output_folders:
    if os.path.isdir(folder):
        shutil.rmtree(folder)

# Remove all old ZIP files
for zip_path in glob.glob("/kaggle/working/*.zip"):
    try:
        os.remove(zip_path)
    except OSError:
        pass

print("Old output folders and duplicate ZIP files removed.")


# ============================================================
# 3. INPUT FILE PATHS
# ============================================================

BASEFLOW_PATH = (
    "/kaggle/input/datasets/kausar15027/"
    "recharge-comparison-dataset/Baseflow_Time_Series_CSV.csv"
)

WTF_PATH = (
    "/kaggle/input/datasets/kausar15027/"
    "recharge-comparison-dataset/Recharge_WTF_Timeseries_CSV.csv"
)

CLOSE_PATH = (
    "/kaggle/input/datasets/kausar15027/"
    "recharge-comparison-dataset/Station_Closeness_CSV.csv"
)

DROUGHT_PATH = (
    "/kaggle/input/datasets/kausar15027/"
    "indices-drought/SPI_SPEI_W.csv"
)


# ============================================================
# 4. OUTPUT PATHS
# ============================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "annual_recharge_drought_comparison_corrected"
)

WTF_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "WTF_vs_Yearly_SPI_SPEI"
)

STREAMFLOW_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "Streamflow_vs_Yearly_SPI_SPEI"
)

DATA_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "Arranged_Annual_Data"
)

DIAGNOSTIC_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "Diagnostics"
)

for folder in [
    OUTPUT_DIR,
    WTF_OUTPUT_DIR,
    STREAMFLOW_OUTPUT_DIR,
    DATA_OUTPUT_DIR,
    DIAGNOSTIC_OUTPUT_DIR
]:
    os.makedirs(folder, exist_ok=True)


# ============================================================
# 5. ANALYSIS SETTINGS
# ============================================================

START_YEAR = 2010
END_YEAR = 2022

YEARS = np.arange(
    START_YEAR,
    END_YEAR + 1
)

streamflow_methods = [
    "Chapman",
    "Eckhardt",
    "LH",
    "Recession Analysis"
]


# ============================================================
# 6. PLOT STYLES
# ============================================================

METHOD_STYLES = {
    "Chapman": {
        "color": "#1f77b4",
        "marker": "o",
        "linestyle": "-",
        "linewidth": 1.4
    },
    "Eckhardt": {
        "color": "#ff7f0e",
        "marker": "s",
        "linestyle": "-",
        "linewidth": 1.4
    },
    "LH": {
        "color": "#2ca02c",
        "marker": "^",
        "linestyle": "-",
        "linewidth": 1.4
    },
    "Recession Analysis": {
        "color": "#d62728",
        "marker": "D",
        "linestyle": "-",
        "linewidth": 1.4
    }
}

DROUGHT_STYLES = {
    "SPI": {
        "color": "#6a3d9a",
        "marker": "o",
        "linestyle": "--",
        "linewidth": 1.3
    },
    "SPEI": {
        "color": "#17becf",
        "marker": "s",
        "linestyle": "-.",
        "linewidth": 1.3
    }
}

WTF_COLORS = [
    "#005f73",
    "#0a9396",
    "#3a86ff",
    "#4361ee",
    "#7209b7",
    "#b5179e",
    "#f72585",
    "#e76f51",
    "#2a9d8f",
    "#6c757d"
]


# ============================================================
# 7. CHECK INPUT FILES
# ============================================================

file_paths = {
    "Baseflow dataset": BASEFLOW_PATH,
    "WTF dataset": WTF_PATH,
    "Station-closeness dataset": CLOSE_PATH,
    "SPI/SPEI dataset": DROUGHT_PATH
}

missing_files = []

print("\nINPUT FILE CHECK")
print("=" * 70)

for name, path in file_paths.items():

    exists = os.path.exists(path)

    print(f"{name}: {exists}")
    print(path)
    print("-" * 70)

    if not exists:
        missing_files.append(name)

if missing_files:
    raise FileNotFoundError(
        "The following files were not found: "
        + ", ".join(missing_files)
    )


# ============================================================
# 8. READ DATA
# ============================================================

baseflow = pd.read_csv(BASEFLOW_PATH)
wtf = pd.read_csv(WTF_PATH)
close = pd.read_csv(CLOSE_PATH)
drought = pd.read_csv(DROUGHT_PATH)

for dataframe in [
    baseflow,
    wtf,
    close,
    drought
]:
    dataframe.columns = (
        dataframe.columns
        .astype(str)
        .str.strip()
    )

print("\nDATASET COLUMNS")
print("=" * 70)

print("Baseflow:", baseflow.columns.tolist())
print("WTF:", wtf.columns.tolist())
print("Closeness:", close.columns.tolist())
print("Drought:", drought.columns.tolist())


# ============================================================
# 9. HELPER FUNCTIONS
# ============================================================

def clean_filename(value):
    """
    Convert station names or IDs into safe filenames.
    """

    value = str(value).strip()

    value = re.sub(
        r'[\\/*?:"<>|]',
        "_",
        value
    )

    return value.replace(" ", "_")


def find_column(dataframe, possible_names):
    """
    Find a dataframe column using case-insensitive matching.
    """

    lookup = {
        str(column).strip().lower(): column
        for column in dataframe.columns
    }

    for possible_name in possible_names:

        key = str(possible_name).strip().lower()

        if key in lookup:
            return lookup[key]

    return None


def standardize_id_column(series):
    """
    Standardize station identifiers and remove trailing .0.
    """

    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )


def format_year_axis(axis):
    """
    Format the x-axis and primary y-axis.
    """

    axis.set_xlim(
        START_YEAR - 0.3,
        END_YEAR + 0.3
    )

    axis.set_xticks(YEARS)

    axis.tick_params(
        axis="x",
        rotation=45,
        labelsize=10,
        width=1
    )

    axis.tick_params(
        axis="y",
        labelsize=10,
        width=1
    )

    axis.set_xlabel(
        "Year",
        fontsize=12,
        fontweight="bold"
    )

    axis.grid(
        True,
        which="major",
        axis="both",
        linestyle="--",
        linewidth=0.6,
        alpha=0.30
    )

    axis.set_axisbelow(True)

    for spine in axis.spines.values():
        spine.set_linewidth(1)


def combine_legends(
    left_axis,
    right_axis,
    location="upper right",
    columns=1
):
    """
    Combine legends from both y-axes.
    """

    left_lines, left_labels = (
        left_axis.get_legend_handles_labels()
    )

    right_lines, right_labels = (
        right_axis.get_legend_handles_labels()
    )

    all_lines = left_lines + right_lines
    all_labels = left_labels + right_labels

    legend = left_axis.legend(
        all_lines,
        all_labels,
        loc=location,
        fontsize=8.5,
        ncol=columns,
        frameon=True,
        fancybox=True,
        framealpha=0.95,
        borderpad=0.7,
        labelspacing=0.5,
        handlelength=2.4
    )

    legend.get_frame().set_edgecolor("#b0b0b0")
    legend.get_frame().set_linewidth(0.8)


def add_drought_series(
    right_axis,
    annual_drought
):
    """
    Plot yearly mean SPI and SPEI on the right y-axis.
    """

    spi_style = DROUGHT_STYLES["SPI"]
    spei_style = DROUGHT_STYLES["SPEI"]

    right_axis.plot(
        annual_drought["Year"],
        annual_drought["SPI"],
        color=spi_style["color"],
        marker=spi_style["marker"],
        linestyle=spi_style["linestyle"],
        linewidth=spi_style["linewidth"],
        markersize=4.5,
        markerfacecolor="white",
        markeredgecolor=spi_style["color"],
        markeredgewidth=1,
        label="Yearly mean SPI",
        zorder=6
    )

    right_axis.plot(
        annual_drought["Year"],
        annual_drought["SPEI"],
        color=spei_style["color"],
        marker=spei_style["marker"],
        linestyle=spei_style["linestyle"],
        linewidth=spei_style["linewidth"],
        markersize=4.5,
        markerfacecolor="white",
        markeredgecolor=spei_style["color"],
        markeredgewidth=1,
        label="Yearly mean SPEI",
        zorder=6
    )

    right_axis.axhline(
        y=0,
        color="#555555",
        linestyle=":",
        linewidth=1,
        alpha=0.8,
        zorder=2
    )

    # Light wet-condition background
    right_axis.axhspan(
        0,
        3,
        color="#d8f3dc",
        alpha=0.08,
        zorder=0
    )

    # Light dry-condition background
    right_axis.axhspan(
        -3,
        0,
        color="#ffe5d9",
        alpha=0.08,
        zorder=0
    )

    right_axis.set_ylabel(
        "Yearly mean drought index",
        fontsize=12,
        fontweight="bold"
    )

    right_axis.set_ylim(-3, 3)

    right_axis.set_yticks(
        np.arange(
            -3,
            3.1,
            0.5
        )
    )

    right_axis.tick_params(
        axis="y",
        labelsize=10,
        width=1
    )

    right_axis.spines["right"].set_linewidth(1)


# ============================================================
# 10. DETECT DROUGHT COLUMNS
# ============================================================

# The column named Year contains weekly dates such as 1/4/2010.
date_column = find_column(
    drought,
    [
        "Date",
        "Time",
        "Datetime",
        "Week",
        "Year"
    ]
)

spi_column = find_column(
    drought,
    ["SPI"]
)

spei_column = find_column(
    drought,
    ["SPEI"]
)

if date_column is None:
    raise ValueError(
        "No weekly date column was found."
    )

if spi_column is None:
    raise ValueError(
        "SPI column was not found."
    )

if spei_column is None:
    raise ValueError(
        "SPEI column was not found."
    )

print("\nDETECTED DROUGHT COLUMNS")
print("=" * 70)

print("Weekly date column:", date_column)
print("SPI column:", spi_column)
print("SPEI column:", spei_column)


# ============================================================
# 11. PREPARE WEEKLY DROUGHT DATA
# ============================================================

drought["Original_Date"] = (
    drought[date_column]
    .astype(str)
    .str.strip()
)

# Dates follow month/day/year format
try:

    drought["Parsed_Date"] = pd.to_datetime(
        drought["Original_Date"],
        format="mixed",
        errors="coerce",
        dayfirst=False
    )

except TypeError:

    drought["Parsed_Date"] = pd.to_datetime(
        drought["Original_Date"],
        errors="coerce",
        dayfirst=False
    )

failed_dates = drought[
    drought["Parsed_Date"].isna()
].copy()

print("\nDATE PARSING")
print("=" * 70)

print("Total rows:", len(drought))
print(
    "Successfully parsed:",
    drought["Parsed_Date"].notna().sum()
)
print("Failed dates:", len(failed_dates))

if not failed_dates.empty:

    failed_dates[
        ["Original_Date"]
    ].head(50).to_csv(
        os.path.join(
            DIAGNOSTIC_OUTPUT_DIR,
            "Failed_Drought_Dates.csv"
        ),
        index=False
    )

drought["Year_Number"] = (
    drought["Parsed_Date"]
    .dt.year
)

drought["SPI_numeric"] = pd.to_numeric(
    drought[spi_column],
    errors="coerce"
)

drought["SPEI_numeric"] = pd.to_numeric(
    drought[spei_column],
    errors="coerce"
)

drought_clean = drought.dropna(
    subset=[
        "Parsed_Date",
        "Year_Number"
    ]
).copy()

drought_clean["Year_Number"] = (
    drought_clean["Year_Number"]
    .astype(int)
)

drought_clean = drought_clean[
    drought_clean["Year_Number"].between(
        START_YEAR,
        END_YEAR,
        inclusive="both"
    )
].copy()

drought_clean = drought_clean.sort_values(
    "Parsed_Date"
)

drought_clean[
    [
        "Original_Date",
        "Parsed_Date",
        "Year_Number",
        "SPI_numeric",
        "SPEI_numeric"
    ]
].to_csv(
    os.path.join(
        DATA_OUTPUT_DIR,
        "Cleaned_Weekly_SPI_SPEI_2010_2022.csv"
    ),
    index=False
)

print("\nValid weekly drought observations:")
print(len(drought_clean))

print("\nAvailable years:")
print(
    sorted(
        drought_clean["Year_Number"].unique()
    )
)


# ============================================================
# 12. CALCULATE YEARLY MEAN SPI AND SPEI
# ============================================================

annual_drought = (
    drought_clean
    .groupby(
        "Year_Number",
        as_index=False
    )
    .agg(
        SPI=(
            "SPI_numeric",
            "mean"
        ),
        SPEI=(
            "SPEI_numeric",
            "mean"
        ),
        SPI_Weeks=(
            "SPI_numeric",
            "count"
        ),
        SPEI_Weeks=(
            "SPEI_numeric",
            "count"
        ),
        SPI_Minimum=(
            "SPI_numeric",
            "min"
        ),
        SPI_Maximum=(
            "SPI_numeric",
            "max"
        ),
        SPEI_Minimum=(
            "SPEI_numeric",
            "min"
        ),
        SPEI_Maximum=(
            "SPEI_numeric",
            "max"
        )
    )
    .rename(
        columns={
            "Year_Number": "Year"
        }
    )
    .sort_values("Year")
)

all_years = pd.DataFrame(
    {
        "Year": YEARS
    }
)

annual_drought = all_years.merge(
    annual_drought,
    on="Year",
    how="left"
)

annual_drought_path = os.path.join(
    DATA_OUTPUT_DIR,
    "Yearly_Mean_SPI_SPEI_2010_2022.csv"
)

annual_drought.to_csv(
    annual_drought_path,
    index=False
)

print("\nYEARLY MEAN SPI AND SPEI")
print("=" * 70)

display(
    annual_drought[
        [
            "Year",
            "SPI",
            "SPEI",
            "SPI_Weeks",
            "SPEI_Weeks"
        ]
    ]
)

if annual_drought[
    ["SPI", "SPEI"]
].isna().all().all():

    raise ValueError(
        "All yearly SPI and SPEI values are missing."
    )


# ============================================================
# 13. CREATE DROUGHT CHECK PLOT
# ============================================================

fig, check_axis = plt.subplots(
    figsize=(12, 5)
)

spi_style = DROUGHT_STYLES["SPI"]
spei_style = DROUGHT_STYLES["SPEI"]

check_axis.plot(
    annual_drought["Year"],
    annual_drought["SPI"],
    color=spi_style["color"],
    marker=spi_style["marker"],
    linestyle=spi_style["linestyle"],
    linewidth=spi_style["linewidth"],
    markersize=4.5,
    markerfacecolor="white",
    markeredgecolor=spi_style["color"],
    markeredgewidth=1,
    label="Yearly mean SPI"
)

check_axis.plot(
    annual_drought["Year"],
    annual_drought["SPEI"],
    color=spei_style["color"],
    marker=spei_style["marker"],
    linestyle=spei_style["linestyle"],
    linewidth=spei_style["linewidth"],
    markersize=4.5,
    markerfacecolor="white",
    markeredgecolor=spei_style["color"],
    markeredgewidth=1,
    label="Yearly mean SPEI"
)

check_axis.axhline(
    y=0,
    color="#555555",
    linestyle=":",
    linewidth=1
)

check_axis.set_xlim(
    START_YEAR - 0.3,
    END_YEAR + 0.3
)

check_axis.set_xticks(YEARS)
check_axis.set_ylim(-3, 3)

check_axis.set_xlabel(
    "Year",
    fontsize=12,
    fontweight="bold"
)

check_axis.set_ylabel(
    "Yearly mean drought index",
    fontsize=12,
    fontweight="bold"
)

check_axis.set_title(
    "Yearly Mean SPI and SPEI",
    fontsize=14,
    fontweight="bold"
)

check_axis.tick_params(
    axis="x",
    rotation=45
)

check_axis.grid(
    True,
    linestyle="--",
    linewidth=0.6,
    alpha=0.30
)

check_axis.legend(
    loc="upper right",
    frameon=True
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        DIAGNOSTIC_OUTPUT_DIR,
        "Yearly_SPI_SPEI_Check_Plot.png"
    ),
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.close()


# ============================================================
# 14. VALIDATE RECHARGE COLUMNS
# ============================================================

required_baseflow_columns = [
    "Station_ID",
    "Year"
]

required_wtf_columns = [
    "GW_Station",
    "Year",
    "Recharge (mm)"
]

required_close_columns = [
    "Station_ID",
    "GW_Station"
]

for column in required_baseflow_columns:

    if column not in baseflow.columns:
        raise ValueError(
            f"'{column}' was not found "
            "in the baseflow dataset."
        )

for column in required_wtf_columns:

    if column not in wtf.columns:
        raise ValueError(
            f"'{column}' was not found "
            "in the WTF dataset."
        )

for column in required_close_columns:

    if column not in close.columns:
        raise ValueError(
            f"'{column}' was not found "
            "in the closeness dataset."
        )


# ============================================================
# 15. STANDARDIZE STATION IDENTIFIERS
# ============================================================

baseflow["Station_ID"] = standardize_id_column(
    baseflow["Station_ID"]
)

close["Station_ID"] = standardize_id_column(
    close["Station_ID"]
)

wtf["GW_Station"] = standardize_id_column(
    wtf["GW_Station"]
)

close["GW_Station"] = standardize_id_column(
    close["GW_Station"]
)

invalid_identifiers = [
    "",
    "nan",
    "none",
    "null"
]

close = close[
    ~close["Station_ID"]
    .str.lower()
    .isin(invalid_identifiers)
].copy()

close = close[
    ~close["GW_Station"]
    .str.lower()
    .isin(invalid_identifiers)
].copy()


# ============================================================
# 16. PREPARE ANNUAL RECHARGE DATA
# ============================================================

baseflow["Year"] = pd.to_numeric(
    baseflow["Year"],
    errors="coerce"
)

wtf["Year"] = pd.to_numeric(
    wtf["Year"],
    errors="coerce"
)

baseflow = baseflow.dropna(
    subset=["Year"]
).copy()

wtf = wtf.dropna(
    subset=["Year"]
).copy()

baseflow["Year"] = (
    baseflow["Year"]
    .astype(int)
)

wtf["Year"] = (
    wtf["Year"]
    .astype(int)
)

baseflow = baseflow[
    baseflow["Year"].between(
        START_YEAR,
        END_YEAR,
        inclusive="both"
    )
].copy()

wtf = wtf[
    wtf["Year"].between(
        START_YEAR,
        END_YEAR,
        inclusive="both"
    )
].copy()

wtf["Recharge (mm)"] = pd.to_numeric(
    wtf["Recharge (mm)"],
    errors="coerce"
)


# ============================================================
# 17. IDENTIFY AVAILABLE STREAMFLOW METHODS
# ============================================================

available_methods = []

for method in streamflow_methods:

    if method in baseflow.columns:

        baseflow[method] = pd.to_numeric(
            baseflow[method],
            errors="coerce"
        )

        available_methods.append(method)

    else:

        print(
            f"Warning: '{method}' was not found."
        )

if not available_methods:

    raise ValueError(
        "No streamflow-based methods were found."
    )

print("\nAvailable streamflow methods:")
print(available_methods)


# ============================================================
# 18. CREATE CATCHMENT PLOTS
# ============================================================

station_ids = (
    close["Station_ID"]
    .dropna()
    .unique()
)

plot_summary = []

print("\nNumber of catchments:", len(station_ids))


for station_id in station_ids:

    print(
        f"Processing catchment: {station_id}"
    )

    safe_station_id = clean_filename(
        station_id
    )

    matched_gw_stations = (
        close.loc[
            close["Station_ID"] == station_id,
            "GW_Station"
        ]
        .dropna()
        .unique()
    )

    current_wtf = wtf[
        wtf["GW_Station"].isin(
            matched_gw_stations
        )
    ].copy()

    current_baseflow = baseflow[
        baseflow["Station_ID"] == station_id
    ].copy()


    # ========================================================
    # 18.1 WTF RECHARGE + SPI/SPEI
    # ========================================================

    if not current_wtf.empty:

        annual_wtf = (
            current_wtf
            .groupby(
                [
                    "Year",
                    "GW_Station"
                ],
                as_index=False
            )
            .agg(
                WTF_Recharge_mm=(
                    "Recharge (mm)",
                    "mean"
                )
            )
            .sort_values(
                [
                    "GW_Station",
                    "Year"
                ]
            )
        )

        wtf_drought_export = annual_wtf.merge(
            annual_drought[
                [
                    "Year",
                    "SPI",
                    "SPEI",
                    "SPI_Weeks",
                    "SPEI_Weeks"
                ]
            ],
            on="Year",
            how="left"
        )

        wtf_drought_export.to_csv(
            os.path.join(
                DATA_OUTPUT_DIR,
                f"Annual_WTF_SPI_SPEI_"
                f"{safe_station_id}.csv"
            ),
            index=False
        )

        fig, left_axis = plt.subplots(
            figsize=(15, 7)
        )

        fig.patch.set_facecolor("white")
        left_axis.set_facecolor("#fcfcfc")

        for station_number, (
            gw_station,
            station_data
        ) in enumerate(
            annual_wtf.groupby("GW_Station")
        ):

            station_color = WTF_COLORS[
                station_number % len(WTF_COLORS)
            ]

            left_axis.plot(
                station_data["Year"],
                station_data["WTF_Recharge_mm"],
                color=station_color,
                marker="o",
                linestyle="-",
                linewidth=1.4,
                markersize=4.5,
                markerfacecolor="white",
                markeredgecolor=station_color,
                markeredgewidth=1,
                label=f"WTF: {gw_station}",
                zorder=7
            )

        left_axis.set_ylabel(
            "WTF recharge (mm)",
            fontsize=12,
            fontweight="bold"
        )

        format_year_axis(left_axis)

        right_axis = left_axis.twinx()

        add_drought_series(
            right_axis,
            annual_drought
        )

        combine_legends(
            left_axis,
            right_axis,
            location="upper right",
            columns=1
        )

        left_axis.set_title(
            "Annual WTF Recharge and "
            "Yearly Mean Drought Indices\n"
            f"Discharge Station/Catchment: {station_id}",
            fontsize=15,
            fontweight="bold",
            pad=15
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                WTF_OUTPUT_DIR,
                f"Annual_WTF_SPI_SPEI_"
                f"{safe_station_id}.png"
            ),
            dpi=300,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.close()

        wtf_plot_created = True

    else:

        print(
            f"No WTF data for catchment {station_id}."
        )

        wtf_plot_created = False


    # ========================================================
    # 18.2 STREAMFLOW METHODS + SPI/SPEI
    # ========================================================

    if not current_baseflow.empty:

        annual_baseflow = (
            current_baseflow
            .groupby(
                "Year",
                as_index=False
            )[available_methods]
            .mean()
            .sort_values("Year")
        )

        streamflow_drought_export = (
            annual_baseflow.merge(
                annual_drought[
                    [
                        "Year",
                        "SPI",
                        "SPEI",
                        "SPI_Weeks",
                        "SPEI_Weeks"
                    ]
                ],
                on="Year",
                how="left"
            )
        )

        streamflow_drought_export.to_csv(
            os.path.join(
                DATA_OUTPUT_DIR,
                f"Annual_Streamflow_SPI_SPEI_"
                f"{safe_station_id}.csv"
            ),
            index=False
        )

        fig, left_axis = plt.subplots(
            figsize=(15, 7)
        )

        fig.patch.set_facecolor("white")
        left_axis.set_facecolor("#fcfcfc")

        for method in available_methods:

            method_style = METHOD_STYLES.get(
                method,
                {
                    "color": "#333333",
                    "marker": "o",
                    "linestyle": "-",
                    "linewidth": 1.4
                }
            )

            left_axis.plot(
                annual_baseflow["Year"],
                annual_baseflow[method],
                color=method_style["color"],
                marker=method_style["marker"],
                linestyle=method_style["linestyle"],
                linewidth=method_style["linewidth"],
                markersize=4.5,
                markerfacecolor="white",
                markeredgecolor=method_style["color"],
                markeredgewidth=1,
                label=method,
                zorder=7
            )

        left_axis.set_ylabel(
            "Streamflow-based recharge (mm)",
            fontsize=12,
            fontweight="bold"
        )

        format_year_axis(left_axis)

        right_axis = left_axis.twinx()

        add_drought_series(
            right_axis,
            annual_drought
        )

        combine_legends(
            left_axis,
            right_axis,
            location="upper right",
            columns=2
        )

        left_axis.set_title(
            "Annual Streamflow-Based Recharge and "
            "Yearly Mean Drought Indices\n"
            f"Discharge Station/Catchment: {station_id}",
            fontsize=15,
            fontweight="bold",
            pad=15
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                STREAMFLOW_OUTPUT_DIR,
                f"Annual_Streamflow_SPI_SPEI_"
                f"{safe_station_id}.png"
            ),
            dpi=300,
            bbox_inches="tight",
            facecolor="white"
        )

        plt.close()

        streamflow_plot_created = True

    else:

        print(
            f"No streamflow data for catchment {station_id}."
        )

        streamflow_plot_created = False


    plot_summary.append(
        {
            "Station_ID": station_id,
            "Number_of_Matched_GW_Stations": (
                len(matched_gw_stations)
            ),
            "WTF_Plot_Created": (
                wtf_plot_created
            ),
            "Streamflow_Plot_Created": (
                streamflow_plot_created
            )
        }
    )


# ============================================================
# 19. SAVE PROCESSING SUMMARY
# ============================================================

plot_summary_df = pd.DataFrame(
    plot_summary
)

plot_summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Plot_Processing_Summary.csv"
    ),
    index=False
)

print("\nPROCESSING SUMMARY")
print("=" * 70)

display(plot_summary_df)


# ============================================================
# 20. CREATE ONLY ONE CORRECTED ZIP FILE
# ============================================================

ZIP_BASE = (
    "/kaggle/working/"
    "Corrected_Annual_Recharge_SPI_SPEI_2010_2022"
)

zip_file = shutil.make_archive(
    ZIP_BASE,
    "zip",
    OUTPUT_DIR
)

# Safety check: remove any ZIP except the corrected final ZIP
for old_zip in glob.glob("/kaggle/working/*.zip"):

    if (
        os.path.abspath(old_zip)
        != os.path.abspath(zip_file)
    ):

        try:
            os.remove(old_zip)
        except OSError:
            pass


print("\n" + "=" * 70)
print("Processing completed successfully.")

print("\nFinal output folder:")
print(OUTPUT_DIR)

print("\nOnly final ZIP retained:")
print(zip_file)

print("=" * 70)


# Kaggle download link
FileLink(zip_file)